In [6]:
import os
import pandas as pd
import math # 결측치 체크용
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()
SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

CSV_PATH = "./nikke_arts/metadata.csv"

def migrate_csv_to_supabase(csv_path):
    print(f"[마이그레이션 시작] {csv_path} 파일을 읽어옵니다...")
    
    df = pd.read_csv(csv_path)
    total_records = len(df)
    
    if 'filename' in df.columns:
        df = df.drop(columns=['filename'])
    df['post_uuid'] = df['post_uuid'].astype(str)
    
    # 순수 파이썬 딕셔너리.
    records = df.to_dict('records')
    
    # 정수형으로 바꿀 컬럼들
    int_columns = [
        'browse_count', 'upvote_count', 'collection_count', 
        'comment_count', 'pic_click_count', 'forward_count', 'ai_flag'
    ]
    
    clean_records = []
    
    # 순수 파이썬 레벨에서 타입을 검사하고 캐스팅합니다.
    for row in records:
        clean_row = {}
        for key, value in row.items():
            # Pandas의 결측치(NaN, NaT)인지 확인
            if pd.isna(value):
                clean_row[key] = None
            elif key in int_columns:
                # float로 바꾸고 int로
                clean_row[key] = int(float(value))
            elif key == 'created_at':
                clean_row[key] = float(value) # SQL에서 numeric으로 잡았으므로 float 유지
            elif key in ['is_original', 'is_official']:
                clean_row[key] = bool(value)
            else:
                clean_row[key] = str(value) if value is not None else None
                
        clean_records.append(clean_row)
        
    chunk_size = 500
    successful_inserts = 0
    
    print(f"총 {total_records}개의 데이터를 {chunk_size}개씩 쪼개서 업로드합니다.")
    print("-" * 50)
    
    for i in range(0, total_records, chunk_size):
        chunk = clean_records[i : i + chunk_size] 
        
        try:
            res = supabase.table("nikke_arts").upsert(chunk).execute()
            successful_inserts += len(chunk)
            print(f"[{i} ~ {i + len(chunk) - 1}] 배치 업로드 성공! (누적: {successful_inserts}/{total_records})")
        except Exception as e:
            print(f"에러 발생 구간 [{i} ~ {i + len(chunk) - 1}]: {e}")
            break
            
    print("-" * 50)
    print(f"[마이그레이션 완료] 총 {successful_inserts}건의 데이터가 Supabase에 적재되었습니다.")

# 실행
migrate_csv_to_supabase(CSV_PATH)

[마이그레이션 시작] ./nikke_arts/metadata.csv 파일을 읽어옵니다...
총 3536개의 데이터를 500개씩 쪼개서 업로드합니다.
--------------------------------------------------
[0 ~ 499] 배치 업로드 성공! (누적: 500/3536)
[500 ~ 999] 배치 업로드 성공! (누적: 1000/3536)
[1000 ~ 1499] 배치 업로드 성공! (누적: 1500/3536)
[1500 ~ 1999] 배치 업로드 성공! (누적: 2000/3536)
[2000 ~ 2499] 배치 업로드 성공! (누적: 2500/3536)
[2500 ~ 2999] 배치 업로드 성공! (누적: 3000/3536)
[3000 ~ 3499] 배치 업로드 성공! (누적: 3500/3536)
[3500 ~ 3535] 배치 업로드 성공! (누적: 3536/3536)
--------------------------------------------------
[마이그레이션 완료] 총 3536건의 데이터가 Supabase에 적재되었습니다.
